# Joint Kaggle benchmark: Research Conditional DDPM vs Fully-Gated DAE

## Goal

This single Kaggle notebook performs the complete comparison pipeline without modifying the original 12-lead LUDB source code:

1. Clone and pin [`AhmedAShaheen/fully_gated_DAE`](https://github.com/AhmedAShaheen/fully_gated_DAE.git).
2. Generate the official random-mixed-noise datasets `Dataset_1.pkl` and `Dataset_2.pkl`.
3. Train/resume the proposed Conditional DDPM as a one-lead benchmark adaptation.
4. Run deterministic DDIM inference for the proposed model.
5. Run the published TensorFlow checkpoints for the baseline architectures.
6. Recalculate every model with one shared implementation of SSD, MAD, PRD, cosine similarity, RMSE, MAE, and SNR.
7. Export segment-level evidence, subject-level comparisons, Wilcoxon tests, tables, and charts.

The core research architecture is preserved: multi-scale HNF blocks, FiLM timestep conditioning, bottleneck self-attention, U-Net encoder/decoder, and the multi-domain diffusion objective. Only the interface changes from 12 leads to one lead: `24→2` input channels and `12→1` output channel.

## Important Kaggle workflow

A full 400-epoch training run for two 11.5M-parameter diffusion models will usually exceed one Kaggle session. This notebook is therefore phase-controlled and resumable:

- **Training versions:** keep `RUN_TRAIN=True`. Each session advances both `nv1` and `nv2` by `MAX_EPOCHS_PER_SPLIT_THIS_RUN` epochs.
- Save a Kaggle version after every session, then attach that version's output as a Kaggle Dataset/Input before continuing.
- **Evaluation version:** after both best checkpoints exist, set `RUN_TRAIN=False` and enable research inference, baseline inference, and evaluation.

For official results, keep `MAX_TEST_SEGMENTS=None`. Any finite value is a smoke test and must not be reported as the benchmark result.

## 1. Install and import dependencies

In [ ]:
!pip -q install "wfdb>=4.1,<5" "PyWavelets>=1.5" "seaborn>=0.13" "prettytable>=3.9"

In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import math
import os
import pickle
import random
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE.type != "cuda":
    print("WARNING: Enable a GPU accelerator before training or inference.")

## 2. Reproducible configuration

Change only the phase switches and session epoch limit during normal operation. Scientific configuration is stored inside every new checkpoint and checked during resume.

In [ ]:
# =========================
# PHASE SWITCHES
# =========================
RUN_DATA_PREP = True
RUN_TRAIN = True
TRAIN_SPLITS = [1, 2]
RUN_RESEARCH_INFERENCE = False
RUN_BASELINE_INFERENCE = False
RUN_COMMON_EVALUATION = False

# A full official test uses None. Use 128 only for a smoke test.
MAX_TEST_SEGMENTS = None
SAVE_PREDICTIONS = False

# Training is intentionally bounded per Kaggle session and resumes later.
TARGET_EPOCHS = 400
MAX_EPOCHS_PER_SPLIT_THIS_RUN = 10

# Storage policy: training persists only .pth checkpoints.
FGDAE_REPO_URL = "https://github.com/AhmedAShaheen/fully_gated_DAE.git"
FGDAE_REPO_COMMIT = "70e768e"
TRAIN_ONLY = RUN_TRAIN and not (RUN_RESEARCH_INFERENCE or RUN_BASELINE_INFERENCE or RUN_COMMON_EVALUATION)
TEMP_ROOT = Path("/kaggle/temp/joint_fgdae_benchmark")
RESULT_ROOT = TEMP_ROOT if TRAIN_ONLY else Path("/kaggle/working/joint_fgdae_benchmark")
FGDAE_REPO_DIR = Path("/kaggle/temp/fully_gated_DAE")
DATA_DIR = TEMP_ROOT / "data" / "4_RMN"
CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
PREDICTION_DIR = RESULT_ROOT / "predictions"
METRIC_DIR = RESULT_ROOT / "metrics"
REPORT_DIR = RESULT_ROOT / "reports"
for directory in (DATA_DIR, CHECKPOINT_DIR, PREDICTION_DIR, METRIC_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Shared experiment settings.
SEED = 1234
SIGNAL_LENGTH = 512
CHANNELS = 1
TRAIN_FRACTION = 0.70
BATCH_SIZE = 96
INFERENCE_BATCH_SIZE = 32
LR = 1e-4
NUM_DIFFUSION_STEPS = 50
BETA_START = 1e-4
BETA_END = 0.5
BETA_SCHEDULE = "quad"
GRAD_CLIP_NORM = 1.0
LR_STEP_SIZE = 150
LR_GAMMA = 0.1
LAMBDA_TIME = 1.0
LAMBDA_FREQ = 0.1
STFT_N_FFT = 128
STFT_HOP_LENGTH = 64
BASE_FEATS = 80
EMB_DIM = 128
DDIM_STEPS = 50
DDIM_ETA = 0.0
DDIM_SHOTS = 1
NUM_WORKERS = 0
PIN_MEMORY = DEVICE.type == "cuda"
PRIMARY_PRD = "paper"  # "paper" or "repo"

BASELINE_MODELS = [
    "Vanilla DAE", "CNN-DAE", "DRNN", "FCN-DAE", "DeepFilter", "ACDAE",
    "CBAM-DAE", "TCDAE", "FGDAE q=1", "FGDAE q=2",
    "FGDAE q=3", "FGDAE q=4",
]

CONFIG = {
    "experiment": "research_conditional_ddpm_on_fgdae_rmn",
    "data_protocol": "fully_gated_DAE_4_RMN_nv1_nv2",
    "fgdae_commit": FGDAE_REPO_COMMIT,
    "seed": SEED,
    "signal_length": SIGNAL_LENGTH,
    "channels": CHANNELS,
    "target_epochs": TARGET_EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "diffusion_steps": NUM_DIFFUSION_STEPS,
    "beta_start": BETA_START,
    "beta_end": BETA_END,
    "beta_schedule": BETA_SCHEDULE,
    "lambda_time": LAMBDA_TIME,
    "lambda_freq": LAMBDA_FREQ,
    "stft_n_fft": STFT_N_FFT,
    "stft_hop_length": STFT_HOP_LENGTH,
    "base_feats": BASE_FEATS,
    "emb_dim": EMB_DIM,
    "model_input_channels": 2,
    "model_output_channels": 1,
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(json.dumps(CONFIG, indent=2))
print("Phase switches:", {
    "train": RUN_TRAIN,
    "research_inference": RUN_RESEARCH_INFERENCE,
    "baseline_inference": RUN_BASELINE_INFERENCE,
    "evaluation": RUN_COMMON_EVALUATION,
})

## 3. Clone the benchmark and prepare RMN datasets

The original download script is not used because it contains syntax/path issues. This cell downloads the two PhysioNet archives directly, applies one local cross-platform path fix, and calls the repository's own `generate_data.py` so the contamination protocol remains authoritative.

If a previous Kaggle output containing both `Dataset_1.pkl` and `Dataset_2.pkl` is attached under `/kaggle/input`, the notebook reuses it instead of rebuilding the data.

In [ ]:
def attached_dataset_directory(preferred_token="joint_fgdae_benchmark"):
    parents = sorted({
        path.parent
        for path in Path("/kaggle/input").rglob("Dataset_1.pkl")
        if (path.parent / "Dataset_2.pkl").exists()
    })
    if not parents:
        return None
    preferred = [path for path in parents if preferred_token in str(path).lower()]
    parents = preferred or parents
    if len(parents) != 1:
        raise RuntimeError(
            "Multiple attached directories contain a Dataset_1/2 pair. "
            "Remove the ambiguous Kaggle inputs:\n" + "\n".join(map(str, parents))
        )
    return parents[0]


def clone_benchmark():
    if not FGDAE_REPO_DIR.exists():
        subprocess.run(["git", "clone", FGDAE_REPO_URL, str(FGDAE_REPO_DIR)], check=True)
    subprocess.run(["git", "checkout", FGDAE_REPO_COMMIT], cwd=FGDAE_REPO_DIR, check=True)

    # Patch only the disposable Kaggle clone; source repository remains untouched.
    prep_file = FGDAE_REPO_DIR / "Data_Preparation" / "data_preparation_overlapped.py"
    lines = prep_file.read_text(encoding="utf-8").splitlines()
    patched = []
    for line in lines:
        if "test_name = re.search" in line:
            indent = line[:len(line) - len(line.lstrip())]
            line = indent + "test_name = str(signal_name).replace('\\\\', '/').split('/')[-1]"
        patched.append(line)
    prep_file.write_text("\n".join(patched) + "\n", encoding="utf-8")


def copy_cached_datasets():
    source_directory = attached_dataset_directory()
    for version in (1, 2):
        destination = DATA_DIR / f"Dataset_{version}.pkl"
        if destination.exists():
            continue
        if source_directory is None:
            return False
        source = source_directory / f"Dataset_{version}.pkl"
        shutil.copy2(source, destination)
        print("Reused:", source, "->", destination)
    return all((DATA_DIR / f"Dataset_{v}.pkl").exists() for v in (1, 2))


def download_archive_with_retry(url, archive_path, attempts=5):
    if archive_path.exists() and zipfile.is_zipfile(archive_path):
        return
    # A failed urlretrieve may leave a partial file; never treat it as a cache hit.
    partial_path = archive_path.with_suffix(archive_path.suffix + '.part')
    for attempt in range(1, attempts + 1):
        try:
            request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 ECG-benchmark/1.0'})
            with urllib.request.urlopen(request, timeout=180) as response, open(partial_path, 'wb') as output:
                shutil.copyfileobj(response, output, length=1024 * 1024)
            with zipfile.ZipFile(partial_path) as archive:
                bad_member = archive.testzip()
                if bad_member is not None:
                    raise zipfile.BadZipFile(f'Corrupt archive member: {bad_member}')
            partial_path.replace(archive_path)
            return
        except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError, OSError, zipfile.BadZipFile) as error:
            if partial_path.exists():
                partial_path.unlink()
            if attempt == attempts:
                raise RuntimeError(
                    f'Download failed after {attempts} attempts: {url}. '
                    'Attach Dataset_1.pkl and Dataset_2.pkl as Kaggle Input to skip PhysioNet downloads.'
                ) from error
            delay = min(30, 3 * 2 ** (attempt - 1))
            print(f'Download attempt {attempt}/{attempts} failed: {error}; retrying in {delay}s')
            time.sleep(delay)


def generate_rmn_datasets():
    clone_benchmark()
    repo_data = FGDAE_REPO_DIR / "data"
    repo_dataset_dir = repo_data / "4_RMN"
    repo_dataset_dir.mkdir(parents=True, exist_ok=True)
    downloads = {
        "qt-database-1.0.0.zip": "https://physionet.org/static/published-projects/qtdb/qt-database-1.0.0.zip",
        "mit-bih-noise-stress-test-database-1.0.0.zip": "https://physionet.org/static/published-projects/nstdb/mit-bih-noise-stress-test-database-1.0.0.zip",
    }
    for archive_name, url in downloads.items():
        archive_path = FGDAE_REPO_DIR / archive_name
        print("Ensuring PhysioNet archive:", url)
        download_archive_with_retry(url, archive_path)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(repo_data)
    subprocess.run(
        [sys.executable, "generate_data.py", "--Data", "RMN1"],
        cwd=FGDAE_REPO_DIR,
        check=True,
    )
    for version in (1, 2):
        shutil.copy2(repo_dataset_dir / f"Dataset_{version}.pkl", DATA_DIR / f"Dataset_{version}.pkl")


if RUN_DATA_PREP:
    cache_complete = copy_cached_datasets()
    if not cache_complete:
        generate_rmn_datasets()
else:
    copy_cached_datasets()

clone_benchmark()  # Also required later for baseline definitions and weights.
DATASET_PATHS = {version: DATA_DIR / f"Dataset_{version}.pkl" for version in (1, 2)}
for version, path in DATASET_PATHS.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing nv{version} dataset: {path}")
    print(f"nv{version}: {path} ({path.stat().st_size / 1024**2:.1f} MB)")

## 4. Validate the shared dataset contract

In [ ]:
dataset_audit = []
for version, path in DATASET_PATHS.items():
    with open(path, "rb") as handle:
        dataset_payload = pickle.load(handle)
    if len(dataset_payload) != 7:
        raise ValueError(f"nv{version}: expected 7 dataset elements, got {len(dataset_payload)}")
    x_train, y_train, x_test, y_test, noise_scale, noise_mask, subject_names = dataset_payload
    if x_train.shape != y_train.shape or x_test.shape != y_test.shape:
        raise ValueError(f"nv{version}: noisy/clean shape mismatch")
    if x_train.shape[1:] != (512, 1) or x_test.shape[1:] != (512, 1):
        raise ValueError(f"nv{version}: expected (N,512,1), got {x_train.shape}, {x_test.shape}")
    if len(noise_scale) != len(x_test) or len(noise_mask) != len(x_test) or len(subject_names) != len(x_test):
        raise ValueError(f"nv{version}: test metadata length mismatch")
    dataset_audit.append({
        "noise_version": f"nv{version}",
        "train_segments": len(x_train),
        "test_segments": len(x_test),
        "shape": str(x_test.shape),
        "dtype": str(x_test.dtype),
        "clean_test_sha256": hashlib.sha256(np.asarray(y_test).tobytes()).hexdigest()[:16],
    })
    del dataset_payload, x_train, y_train, x_test, y_test
    gc.collect()

dataset_audit = pd.DataFrame(dataset_audit)
display(dataset_audit)
dataset_audit.to_csv(REPORT_DIR / "dataset_audit.csv", index=False)

## 5. PyTorch adapter

This adapter performs only the required layout conversion `(N,512,1) → (N,1,512)`. It does not regenerate artifacts, resample, normalize, or inspect the test targets during training.

In [ ]:
class ArrayPairDataset(Dataset):
    def __init__(self, clean, noisy):
        clean = np.asarray(clean, dtype=np.float32)
        noisy = np.asarray(noisy, dtype=np.float32)
        if clean.shape != noisy.shape or clean.shape[1:] != (512, 1):
            raise ValueError((clean.shape, noisy.shape))
        self.clean = torch.from_numpy(clean).permute(0, 2, 1).contiguous()
        self.noisy = torch.from_numpy(noisy).permute(0, 2, 1).contiguous()

    def __len__(self):
        return len(self.clean)

    def __getitem__(self, index):
        return self.clean[index], self.noisy[index]


def load_split_arrays(noise_version):
    with open(DATASET_PATHS[noise_version], "rb") as handle:
        return pickle.load(handle)


sample_payload = load_split_arrays(1)
sample_x, sample_y = sample_payload[0][:2], sample_payload[1][:2]
sample_dataset = ArrayPairDataset(sample_y, sample_x)
sample_clean, sample_noisy = sample_dataset[0]
print("PyTorch clean/noisy:", sample_clean.shape, sample_noisy.shape)
assert sample_clean.shape == sample_noisy.shape == (1, 512)
del sample_payload, sample_dataset, sample_clean, sample_noisy
gc.collect()

## 6. Research architecture — one-lead benchmark adaptation

This is the same advanced U-Net family used by the existing research training notebook. Input/output channel counts are explicitly `2/1`; HNF, FiLM, attention, depth, and feature widths remain unchanged.

In [ ]:
class HNFBlockUNet(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_sizes=(3, 5, 9, 15)):
        super().__init__()
        self.multi_convs = nn.ModuleList([
            nn.Conv1d(in_channels, out_channels // len(kernel_sizes), k, padding=k // 2, padding_mode='reflect')
            for k in kernel_sizes
        ])
        self.agg_conv = nn.Conv1d(out_channels, out_channels, 1)
        self.half_inst_norm = nn.InstanceNorm1d(out_channels // 2)
        self.act = nn.ReLU(inplace=True)
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        out = torch.cat([conv(x) for conv in self.multi_convs], dim=1)
        out = self.agg_conv(out)
        half = out.shape[1] // 2
        out = torch.cat([self.half_inst_norm(out[:, :half, :]), out[:, half:, :]], dim=1)
        out = self.act(out)
        return out + self.residual(x)


class BridgeBlockUNet(nn.Module):
    def __init__(self, features, emb_dim=128):
        super().__init__()
        self.emb_dim = emb_dim
        self.film = nn.Sequential(nn.Linear(emb_dim, features * 2), nn.SiLU())

    def sinusoidal_embedding(self, x):
        x = x.view(-1)
        device = x.device
        half_dim = self.emb_dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x.unsqueeze(-1) * emb.unsqueeze(0)
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)

    def forward(self, x, alpha_bar):
        emb = self.sinusoidal_embedding(alpha_bar)
        scale, shift = self.film(emb).chunk(2, dim=1)
        return x * (1 + scale.unsqueeze(-1)) + shift.unsqueeze(-1)


class SelfAttention1D(nn.Module):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        assert self.head_dim * num_heads == channels
        self.qkv = nn.Conv1d(channels, channels * 3, kernel_size=1)
        self.proj = nn.Conv1d(channels, channels, kernel_size=1)

    def forward(self, x):
        batch, channels, length = x.shape
        qkv = self.qkv(x).reshape(batch, 3, self.num_heads, self.head_dim, length)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        attn = torch.matmul(q.transpose(-2, -1), k) / (self.head_dim ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        out = torch.matmul(attn, v.transpose(-2, -1)).transpose(-2, -1)
        return self.proj(out.reshape(batch, channels, length))


class UNet1D(nn.Module):
    def __init__(self, in_channels=2, base_channels=80, emb_dim=128, out_channels=1):
        super().__init__()
        self.enc1 = HNFBlockUNet(in_channels, base_channels)
        self.bridge1 = BridgeBlockUNet(base_channels, emb_dim)
        self.down1 = nn.Conv1d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1)
        self.enc2 = HNFBlockUNet(base_channels * 2, base_channels * 2)
        self.bridge2 = BridgeBlockUNet(base_channels * 2, emb_dim)
        self.down2 = nn.Conv1d(base_channels * 2, base_channels * 4, kernel_size=4, stride=2, padding=1)
        self.enc3 = HNFBlockUNet(base_channels * 4, base_channels * 4)
        self.bridge3 = BridgeBlockUNet(base_channels * 4, emb_dim)
        self.down3 = nn.Conv1d(base_channels * 4, base_channels * 8, kernel_size=4, stride=2, padding=1)
        self.enc4 = HNFBlockUNet(base_channels * 8, base_channels * 8)
        self.bridge4 = BridgeBlockUNet(base_channels * 8, emb_dim)
        self.attn = SelfAttention1D(base_channels * 8)
        self.up4 = nn.ConvTranspose1d(base_channels * 8, base_channels * 4, kernel_size=4, stride=2, padding=1)
        self.dec4 = HNFBlockUNet(base_channels * 8, base_channels * 4)
        self.up3 = nn.ConvTranspose1d(base_channels * 4, base_channels * 2, kernel_size=4, stride=2, padding=1)
        self.dec3 = HNFBlockUNet(base_channels * 4, base_channels * 2)
        self.up2 = nn.ConvTranspose1d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1)
        self.dec2 = HNFBlockUNet(base_channels * 2, base_channels)
        self.final = nn.Conv1d(base_channels, out_channels, kernel_size=1)

    def forward(self, x, cond, noise_scale):
        inp = torch.cat([x, cond], dim=1)
        e1 = self.bridge1(self.enc1(inp), noise_scale)
        e2 = self.bridge2(self.enc2(self.down1(e1)), noise_scale)
        e3 = self.bridge3(self.enc3(self.down2(e2)), noise_scale)
        e4 = self.bridge4(self.enc4(self.down3(e3)), noise_scale)
        e4 = self.attn(e4)
        d4 = self.dec4(torch.cat([self.up4(e4), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        return self.final(d2)

## 7. Conditional diffusion and multi-domain objective

In [ ]:
def stft_magnitude_loss(prediction, target, continuous_sqrt_alpha):
    batch, channels, length = prediction.shape
    prediction = prediction.reshape(batch * channels, length)
    target = target.reshape(batch * channels, length)
    window = torch.hann_window(STFT_N_FFT, device=prediction.device)
    prediction_stft = torch.stft(
        prediction, n_fft=STFT_N_FFT, hop_length=STFT_HOP_LENGTH,
        window=window, return_complex=True
    )
    target_stft = torch.stft(
        target, n_fft=STFT_N_FFT, hop_length=STFT_HOP_LENGTH,
        window=window, return_complex=True
    )

    prediction_mag = torch.abs(prediction_stft).reshape(batch, channels, *prediction_stft.shape[-2:])
    target_mag = torch.abs(target_stft).reshape(batch, channels, *target_stft.shape[-2:])
    reduce_dims = (1, 2, 3)
    spectral_error = torch.mean((prediction_mag - target_mag) ** 2, dim=reduce_dims)
    target_power = torch.mean(target_mag ** 2, dim=reduce_dims).clamp_min(1e-6)
    relative_error = spectral_error / target_power

    # x0 reconstruction is ill-conditioned at very noisy timesteps. Weight its
    # spectral supervision by diffusion SNR and cap the low-noise weight at 1.
    alpha_bar = continuous_sqrt_alpha.reshape(batch) ** 2
    diffusion_snr = alpha_bar / torch.clamp(1.0 - alpha_bar, min=1e-6)
    frequency_weight = torch.clamp(diffusion_snr, max=1.0)
    weighted_loss = torch.mean(relative_error * frequency_weight)
    return weighted_loss, torch.mean(relative_error), torch.mean(frequency_weight)


def make_beta_schedule(schedule_name, num_steps, start, end):
    if schedule_name == 'linear':
        return torch.linspace(start, end, num_steps)
    if schedule_name == 'quad':
        return torch.linspace(start ** 0.5, end ** 0.5, num_steps) ** 2
    if schedule_name == 'sigmoid':
        values = torch.linspace(-6, 6, num_steps)
        return torch.sigmoid(values) * (end - start) + start
    raise ValueError(schedule_name)


class DDPM(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.model = base_model
        self.num_steps = NUM_DIFFUSION_STEPS
        betas = make_beta_schedule(
            BETA_SCHEDULE, NUM_DIFFUSION_STEPS, BETA_START, BETA_END
        )
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = torch.cat([torch.ones(1), alphas_cumprod[:-1]])
        continuous_boundaries = torch.sqrt(torch.cat([torch.ones(1), alphas_cumprod]))
        posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

        self.register_buffer('betas', betas.float())
        self.register_buffer('alphas_cumprod', alphas_cumprod.float())
        self.register_buffer('alphas_cumprod_prev', alphas_cumprod_prev.float())
        self.register_buffer('continuous_boundaries', continuous_boundaries.float())
        self.register_buffer('posterior_variance', posterior_variance.float())
        self.register_buffer(
            'posterior_log_variance_clipped',
            torch.log(torch.clamp(posterior_variance, min=1e-20)).float(),
        )
        self.register_buffer(
            'posterior_mean_coef1',
            (betas * torch.sqrt(alphas_cumprod_prev) / (1.0 - alphas_cumprod)).float(),
        )
        self.register_buffer(
            'posterior_mean_coef2',
            ((1.0 - alphas_cumprod_prev) * torch.sqrt(alphas) /
             (1.0 - alphas_cumprod)).float(),
        )

    def training_losses(self, clean, noisy_condition):
        batch = clean.shape[0]
        timestep = torch.randint(0, self.num_steps, (batch,), device=clean.device)
        lower = self.continuous_boundaries[timestep]
        upper = self.continuous_boundaries[timestep + 1]
        continuous = lower + torch.rand(batch, device=clean.device) * (upper - lower)
        continuous_3d = continuous.view(batch, 1, 1)

        gaussian_noise = torch.randn_like(clean)
        x_t = (
            continuous_3d * clean
            + torch.sqrt(torch.clamp(1.0 - continuous_3d ** 2, min=0.0)) * gaussian_noise
        )
        predicted_noise = self.model(x_t, noisy_condition, continuous.view(batch, 1))

        # Research B time-domain objective, normalized per element so its
        # scale is independent of batch size and ECG window length.
        loss_time = F.l1_loss(predicted_noise, gaussian_noise, reduction='mean')

        # Use the exact continuous coefficient that generated x_t.
        x0_prediction = (
            x_t
            - torch.sqrt(torch.clamp(1.0 - continuous_3d ** 2, min=0.0)) * predicted_noise
        ) / torch.clamp(continuous_3d, min=1e-6)
        loss_frequency, frequency_unweighted, frequency_weight = stft_magnitude_loss(
            x0_prediction, clean, continuous
        )
        total = LAMBDA_TIME * loss_time + LAMBDA_FREQ * loss_frequency
        return {
            'total': total,
            'time_mean': loss_time,
            'frequency': loss_frequency,
            'frequency_unweighted': frequency_unweighted,
            'frequency_weight': frequency_weight,
            'weighted_frequency': LAMBDA_FREQ * loss_frequency,
        }

    def forward(self, clean, noisy_condition):
        return self.training_losses(clean, noisy_condition)['total']

    def q_posterior(self, x_start, x_t, timestep):
        coef1 = self.posterior_mean_coef1[timestep].view(-1, 1, 1)
        coef2 = self.posterior_mean_coef2[timestep].view(-1, 1, 1)
        mean = coef1 * x_start + coef2 * x_t
        log_variance = self.posterior_log_variance_clipped[timestep].view(-1, 1, 1)
        return mean, log_variance

    @torch.no_grad()
    def sample(self, condition, num_shots=1):
        output_sum = torch.zeros_like(condition)
        batch = condition.shape[0]
        for _ in range(num_shots):
            x = torch.randn_like(condition)
            for step in reversed(range(self.num_steps)):
                timestep = torch.full((batch,), step, device=x.device, dtype=torch.long)
                noise_level = self.continuous_boundaries[step + 1].expand(batch, 1)
                predicted_noise = self.model(x, condition, noise_level)
                alpha_bar = self.alphas_cumprod[step]
                x0_prediction = (
                    x - torch.sqrt(1.0 - alpha_bar) * predicted_noise
                ) / torch.sqrt(alpha_bar)
                posterior_mean, posterior_log_variance = self.q_posterior(
                    x0_prediction, x, timestep
                )
                if step > 0:
                    x = posterior_mean + torch.exp(0.5 * posterior_log_variance) * torch.randn_like(x)
                else:
                    x = posterior_mean
            output_sum += x
        return output_sum / num_shots

## 8. Deterministic DDIM sampler and shared ECG metrics

In [ ]:
@torch.no_grad()
def ddim_sample(model, condition, steps=50, eta=0.0):
    model.eval()
    total_steps = model.num_steps
    steps = min(int(steps), total_steps)
    timestep_sequence = np.unique(np.linspace(0, total_steps - 1, steps, dtype=int)).tolist()
    x = torch.randn_like(condition)
    for sequence_index in reversed(range(len(timestep_sequence))):
        timestep_value = timestep_sequence[sequence_index]
        previous_value = timestep_sequence[sequence_index - 1] if sequence_index > 0 else -1
        batch = x.shape[0]
        noise_level = torch.sqrt(model.alphas_cumprod[timestep_value]).expand(batch, 1)
        predicted_noise = model.model(x, condition, noise_level)
        alpha_t = model.alphas_cumprod[timestep_value]
        alpha_previous = model.alphas_cumprod[previous_value] if previous_value >= 0 else torch.tensor(1.0, device=x.device)
        x0_prediction = (x - torch.sqrt(1.0 - alpha_t) * predicted_noise) / torch.sqrt(alpha_t)
        sigma = eta * torch.sqrt((1.0 - alpha_previous) / (1.0 - alpha_t)) * torch.sqrt(
            torch.clamp(1.0 - alpha_t / alpha_previous, min=0.0)
        )
        direction = torch.sqrt(torch.clamp(1.0 - alpha_previous - sigma ** 2, min=0.0)) * predicted_noise
        random_term = sigma * torch.randn_like(x) if eta > 0 and previous_value >= 0 else 0.0
        x = torch.sqrt(alpha_previous) * x0_prediction + direction + random_term
    return x


def artifact_labels(mask):
    names = np.array(["BW", "MA", "EM"])
    labels = []
    for row in np.asarray(mask).astype(bool):
        active = names[row]
        labels.append("Clean" if len(active) == 0 else "+".join(active))
    return labels


def calculate_segment_metrics(y_true, y_pred, eps=1e-12):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(len(y_true), -1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(len(y_pred), -1)
    if y_true.shape != y_pred.shape:
        raise ValueError((y_true.shape, y_pred.shape))
    error = y_pred - y_true
    squared_error = error ** 2
    ssd = squared_error.sum(axis=1)
    signal_power = (y_true ** 2).sum(axis=1)
    clean_centered = y_true - y_true.mean(axis=1, keepdims=True)
    paper_denominator = (clean_centered ** 2).sum(axis=1)
    repo_denominator = ((y_pred - y_true.mean()) ** 2).sum(axis=1)
    dot = (y_true * y_pred).sum(axis=1)
    norm_product = np.linalg.norm(y_true, axis=1) * np.linalg.norm(y_pred, axis=1)
    return pd.DataFrame({
        "SSD": ssd,
        "MAD_mV": np.abs(error).max(axis=1),
        "PRD_paper_pct": 100.0 * np.sqrt(ssd / np.maximum(paper_denominator, eps)),
        "PRD_repo_pct": 100.0 * np.sqrt(ssd / np.maximum(repo_denominator, eps)),
        "CosSim": dot / np.maximum(norm_product, eps),
        "RMSE_mV": np.sqrt(squared_error.mean(axis=1)),
        "MAE_mV": np.abs(error).mean(axis=1),
        "SNR_dB": 10.0 * np.log10(np.maximum(signal_power, eps) / np.maximum(ssd, eps)),
    })


def decorate_metrics(frame, model_name, version, noise_scale, noise_mask, subject_names):
    frame.insert(0, "segment_index", np.arange(len(frame)))
    frame.insert(0, "subject", list(subject_names)[:len(frame)])
    frame.insert(0, "artifact", artifact_labels(np.asarray(noise_mask)[:len(frame)]))
    frame.insert(0, "noise_scale", np.asarray(noise_scale)[:len(frame)])
    frame.insert(0, "noise_version", f"nv{version}")
    frame.insert(0, "model", model_name)
    return frame

## 9. Resumable training

Training and validation use only `X_train/y_train`. The original ordering is split 70/30 with `shuffle=False`, matching the benchmark code. Test arrays remain untouched until inference.

In [ ]:
def build_research_model():
    base_model = UNet1D(
        in_channels=2,
        base_channels=BASE_FEATS,
        emb_dim=EMB_DIM,
        out_channels=1,
    )
    return DDPM(base_model).to(DEVICE)


def atomic_torch_save(payload, destination):
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    torch.save(payload, temporary)
    temporary.replace(destination)


def load_torch_checkpoint(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:  # PyTorch versions before weights_only was added.
        return torch.load(path, map_location=DEVICE)


def cpu_rng_state(state):
    # torch.load(..., map_location=CUDA) also moves RNG byte tensors to CUDA,
    # while Generator.set_state/torch.set_rng_state require CPU ByteTensor.
    if not torch.is_tensor(state):
        raise TypeError(f"Expected RNG tensor, got {type(state)!r}")
    return state.detach().to(device="cpu", dtype=torch.uint8).contiguous()


def checkpoint_candidates(filename):
    local = CHECKPOINT_DIR / filename
    if local.exists():
        return [local]
    return sorted(Path("/kaggle/input").rglob(filename))


def resolve_checkpoint(filename, required=False):
    candidates = checkpoint_candidates(filename)
    if len(candidates) > 1:
        raise RuntimeError(f"Multiple checkpoints named {filename}:\n" + "\n".join(map(str, candidates)))
    if not candidates:
        if required:
            raise FileNotFoundError(filename)
        return None
    return candidates[0]


@torch.no_grad()
def validation_loss(model, loader):
    model.eval()
    values = []
    for clean, noisy in loader:
        clean = clean.to(DEVICE, non_blocking=True)
        noisy = noisy.to(DEVICE, non_blocking=True)
        values.append(float(model.training_losses(clean, noisy)["total"].item()))
    return float(np.mean(values))


def training_checkpoint_payload(model, optimizer, scheduler, generator, version, epoch, best_value, history):
    return {
        "state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "loader_generator_state": generator.get_state(),
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
        "epoch": epoch,
        "noise_version": version,
        "best_validation": best_value,
        "history": history,
        "config": CONFIG,
    }


def train_noise_version(version):
    payload = load_split_arrays(version)
    x_train, y_train = payload[0], payload[1]
    split_index = int(len(x_train) * TRAIN_FRACTION)
    train_dataset = ArrayPairDataset(y_train[:split_index], x_train[:split_index])
    validation_dataset = ArrayPairDataset(y_train[split_index:], x_train[split_index:])
    generator = torch.Generator().manual_seed(SEED + version)
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, generator=generator,
    )
    validation_loader = DataLoader(
        validation_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )
    del payload, x_train, y_train
    gc.collect()

    model = build_research_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP_SIZE, gamma=LR_GAMMA)
    last_name = f"research_ddpm_rmn_nv{version}_last.pth"
    best_name = f"research_ddpm_rmn_nv{version}_best.pth"
    log_path = REPORT_DIR / f"research_ddpm_rmn_nv{version}_training_log.csv"
    for checkpoint_name in (last_name, best_name):
        prior_path = resolve_checkpoint(checkpoint_name)
        local_path = CHECKPOINT_DIR / checkpoint_name
        if prior_path is not None and prior_path.resolve() != local_path.resolve():
            shutil.copy2(prior_path, local_path)
            print(f"nv{version}: carried forward {checkpoint_name}")
    resume_path = resolve_checkpoint(last_name)
    start_epoch, best_validation, history = 1, float("inf"), []

    if resume_path is not None:
        checkpoint = load_torch_checkpoint(resume_path)
        if checkpoint.get("config") != CONFIG or int(checkpoint.get("noise_version", -1)) != version:
            raise RuntimeError(f"Incompatible resume checkpoint: {resume_path}")
        model.load_state_dict(checkpoint["state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        generator.set_state(cpu_rng_state(checkpoint["loader_generator_state"]))
        random.setstate(checkpoint["python_rng_state"])
        np.random.set_state(checkpoint["numpy_rng_state"])
        torch.set_rng_state(cpu_rng_state(checkpoint["torch_rng_state"]))
        if torch.cuda.is_available() and checkpoint.get("cuda_rng_state_all"):
            torch.cuda.set_rng_state_all([
                cpu_rng_state(state) for state in checkpoint["cuda_rng_state_all"]
            ])
        start_epoch = int(checkpoint["epoch"]) + 1
        best_validation = float(checkpoint["best_validation"])
        history = list(checkpoint.get("history", []))
        print(f"nv{version}: resuming from {resume_path}, epoch {start_epoch}")
    else:
        print(f"nv{version}: starting from scratch")

    stop_epoch = min(TARGET_EPOCHS, start_epoch + MAX_EPOCHS_PER_SPLIT_THIS_RUN - 1)
    if start_epoch > TARGET_EPOCHS:
        print(f"nv{version}: already complete at epoch {start_epoch - 1}")
        return

    for epoch in range(start_epoch, stop_epoch + 1):
        started = time.perf_counter()
        model.train()
        totals, time_losses, frequency_losses = [], [], []
        for clean, noisy in tqdm(train_loader, desc=f"nv{version} epoch {epoch}/{TARGET_EPOCHS}", leave=False):
            clean = clean.to(DEVICE, non_blocking=True)
            noisy = noisy.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            losses = model.training_losses(clean, noisy)
            if not torch.isfinite(losses["total"]):
                raise FloatingPointError(f"Non-finite loss: nv{version}, epoch {epoch}")
            losses["total"].backward()
            gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            if not torch.isfinite(gradient_norm):
                raise FloatingPointError(f"Non-finite gradient: nv{version}, epoch {epoch}")
            optimizer.step()
            totals.append(float(losses["total"].item()))
            time_losses.append(float(losses["time_mean"].item()))
            frequency_losses.append(float(losses["frequency"].item()))
        scheduler.step()
        with torch.random.fork_rng(devices=[torch.cuda.current_device()] if DEVICE.type == "cuda" else []):
            torch.manual_seed(SEED + version * 10000)
            validation_value = validation_loss(model, validation_loader)
        row = {
            "noise_version": version,
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            "train_total": float(np.mean(totals)),
            "train_time": float(np.mean(time_losses)),
            "train_frequency": float(np.mean(frequency_losses)),
            "validation_total": validation_value,
            "seconds": time.perf_counter() - started,
        }
        history.append(row)
        improved = validation_value < best_validation
        best_validation = min(best_validation, validation_value)
        checkpoint = training_checkpoint_payload(
            model, optimizer, scheduler, generator, version, epoch, best_validation, history
        )
        atomic_torch_save(checkpoint, CHECKPOINT_DIR / last_name)
        if improved:
            atomic_torch_save(checkpoint, CHECKPOINT_DIR / best_name)
        pd.DataFrame(history).to_csv(log_path, index=False)
        print(row, "best" if improved else "")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    del model, optimizer, train_loader, validation_loader
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()


if RUN_TRAIN:
    for noise_version in TRAIN_SPLITS:
        train_noise_version(noise_version)

print("Available training outputs:")
for path in sorted(CHECKPOINT_DIR.glob("research_ddpm_rmn_*")):
    print(path.name, f"{path.stat().st_size / 1024**2:.1f} MB")

## 10. Research DDPM inference

Enable only after both `research_ddpm_rmn_nv1_best.pth` and `research_ddpm_rmn_nv2_best.pth` are available locally or attached under `/kaggle/input`.

In [ ]:
def official_or_smoke_length(array):
    return len(array) if MAX_TEST_SEGMENTS is None else min(len(array), int(MAX_TEST_SEGMENTS))


def save_prediction_if_requested(model_name, version, x_test, y_test, y_pred):
    if not SAVE_PREDICTIONS:
        return
    safe_name = model_name.replace(" ", "_").replace("=", "").replace("-", "_")
    np.savez_compressed(
        PREDICTION_DIR / f"predictions_{safe_name}_nv{version}.npz",
        x_test=x_test, y_test=y_test, y_pred=y_pred,
    )


research_runs = []
if RUN_RESEARCH_INFERENCE:
    for version in (1, 2):
        best_path = resolve_checkpoint(f"research_ddpm_rmn_nv{version}_best.pth", required=True)
        checkpoint = load_torch_checkpoint(best_path)
        if checkpoint.get("config") != CONFIG or int(checkpoint.get("noise_version", -1)) != version:
            raise RuntimeError(f"Checkpoint metadata mismatch: {best_path}")
        model = build_research_model()
        model.load_state_dict(checkpoint["state_dict"])
        model.eval()

        payload = load_split_arrays(version)
        x_test, y_test, noise_scale, noise_mask, subject_names = payload[2:]
        count = official_or_smoke_length(x_test)
        x_test = np.asarray(x_test[:count], dtype=np.float32)
        y_test = np.asarray(y_test[:count], dtype=np.float32)
        predictions = []
        torch.manual_seed(SEED + version * 100000)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED + version * 100000)
        started = time.perf_counter()
        for start in tqdm(range(0, count, INFERENCE_BATCH_SIZE), desc=f"MyDDPM nv{version}"):
            condition = torch.from_numpy(x_test[start:start + INFERENCE_BATCH_SIZE]).permute(0, 2, 1).to(DEVICE)
            output = ddim_sample(model, condition, steps=DDIM_STEPS, eta=DDIM_ETA)
            predictions.append(output.permute(0, 2, 1).cpu().numpy())
        elapsed = time.perf_counter() - started
        y_pred = np.concatenate(predictions, axis=0).astype(np.float32)
        if y_pred.shape != y_test.shape or not np.isfinite(y_pred).all():
            raise RuntimeError(f"Invalid MyDDPM prediction nv{version}: {y_pred.shape}")
        metric_frame = calculate_segment_metrics(y_test, y_pred)
        metric_frame = decorate_metrics(
            metric_frame, "MyDDPM", version,
            noise_scale[:count], noise_mask[:count], subject_names[:count],
        )
        metric_frame.to_csv(METRIC_DIR / f"segment_metrics_MyDDPM_nv{version}.csv", index=False)
        save_prediction_if_requested("MyDDPM", version, x_test, y_test, y_pred)
        research_runs.append({
            "model": "MyDDPM",
            "noise_version": f"nv{version}",
            "segments": count,
            "parameters": sum(parameter.numel() for parameter in model.parameters()),
            "inference_seconds": elapsed,
            "segments_per_second": count / elapsed,
            "sampler": "DDIM",
            "steps": DDIM_STEPS,
            "eta": DDIM_ETA,
            "shots": DDIM_SHOTS,
            "official_full_test": MAX_TEST_SEGMENTS is None,
            "checkpoint": str(best_path),
        })
        del model, checkpoint, payload, x_test, y_test, y_pred, predictions
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    pd.DataFrame(research_runs).to_csv(REPORT_DIR / "research_inference_runs.csv", index=False)
    display(pd.DataFrame(research_runs))
else:
    print("Research inference disabled.")

## 11. Published baseline checkpoint inference

This phase imports TensorFlow only after PyTorch work is complete. If GPU memory is insufficient, save the Kaggle version, restart the session, set training/research inference to `False`, and run this same notebook with `RUN_BASELINE_INFERENCE=True`.

In [ ]:
baseline_runs, baseline_failures = [], []
if RUN_BASELINE_INFERENCE:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    import tensorflow as tf
    for gpu in tf.config.list_physical_devices("GPU"):
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass

    if str(FGDAE_REPO_DIR) not in sys.path:
        sys.path.insert(0, str(FGDAE_REPO_DIR))
    previous_directory = Path.cwd()
    os.chdir(FGDAE_REPO_DIR)
    import dl_models as fg_models

    model_specs = {
        "Vanilla DAE": ("Vanilla_DAE", lambda: fg_models.VanillaAutoencoder(signal_size=512)),
        "CNN-DAE": ("CNN_DAE", lambda: fg_models.CNN_DAE(signal_size=512)),
        "DRNN": ("DRNN", lambda: fg_models.DRNN_denoising(signal_size=512)),
        "FCN-DAE": ("FCN_DAE", lambda: fg_models.FCN_DAE(signal_size=512)),
        "DeepFilter": ("DeepFilter", lambda: fg_models.deep_filter_model_I_LANL_dilated(signal_size=512)),
        "ACDAE": ("ACDAE", lambda: fg_models.ECASkipDAE(signal_size=512)),
        "CBAM-DAE": ("CDAE_BAM", lambda: fg_models.AttentionSkipDAE2(signal_size=512)),
        "TCDAE": ("TCDAE", lambda: fg_models.Transformer_DAE(signal_size=512)),
        "FGDAE q=1": ("Proposed_gatedONN1", lambda: fg_models.GatedONNDAE(signal_size=512, q=1)),
        "FGDAE q=2": ("Proposed_gatedONN2", lambda: fg_models.GatedONNDAE(signal_size=512, q=2)),
        "FGDAE q=3": ("Proposed_gatedONN3", lambda: fg_models.GatedONNDAE(signal_size=512, q=3)),
        "FGDAE q=4": ("Proposed_gatedONN4", lambda: fg_models.GatedONNDAE(signal_size=512, q=4)),
    }

    for version in (1, 2):
        payload = load_split_arrays(version)
        x_test, y_test, noise_scale, noise_mask, subject_names = payload[2:]
        count = official_or_smoke_length(x_test)
        x_test = np.asarray(x_test[:count], dtype=np.float32)
        y_test = np.asarray(y_test[:count], dtype=np.float32)
        for model_name in BASELINE_MODELS:
            label, builder = model_specs[model_name]
            checkpoint_path = FGDAE_REPO_DIR / "models" / f"{label}{version}_best.weights.h5"
            try:
                tf.keras.backend.clear_session()
                baseline_model = builder()
                _ = baseline_model(tf.zeros((1, 512, 1), dtype=tf.float32), training=False)
                baseline_model.load_weights(str(checkpoint_path))
                started = time.perf_counter()
                y_pred = baseline_model.predict(x_test, batch_size=INFERENCE_BATCH_SIZE, verbose=0)
                elapsed = time.perf_counter() - started
                y_pred = np.asarray(y_pred, dtype=np.float32).reshape(y_test.shape)
                if not np.isfinite(y_pred).all():
                    raise FloatingPointError("Prediction contains NaN/Inf")
                metric_frame = calculate_segment_metrics(y_test, y_pred)
                metric_frame = decorate_metrics(
                    metric_frame, model_name, version,
                    noise_scale[:count], noise_mask[:count], subject_names[:count],
                )
                safe_name = model_name.replace(" ", "_").replace("=", "").replace("-", "_")
                metric_frame.to_csv(METRIC_DIR / f"segment_metrics_{safe_name}_nv{version}.csv", index=False)
                save_prediction_if_requested(model_name, version, x_test, y_test, y_pred)
                baseline_runs.append({
                    "model": model_name,
                    "noise_version": f"nv{version}",
                    "segments": count,
                    "parameters": int(baseline_model.count_params()),
                    "inference_seconds": elapsed,
                    "segments_per_second": count / elapsed,
                    "official_full_test": MAX_TEST_SEGMENTS is None,
                    "checkpoint": checkpoint_path.name,
                    "tensorflow_version": tf.__version__,
                })
            except Exception as error:
                baseline_failures.append({
                    "model": model_name,
                    "noise_version": f"nv{version}",
                    "error": repr(error),
                })
                print("FAILED", model_name, f"nv{version}", repr(error))
            finally:
                if "baseline_model" in locals():
                    del baseline_model
                if "y_pred" in locals():
                    del y_pred
                tf.keras.backend.clear_session()
                gc.collect()
        del payload, x_test, y_test
        gc.collect()

    os.chdir(previous_directory)
    pd.DataFrame(baseline_runs).to_csv(REPORT_DIR / "baseline_inference_runs.csv", index=False)
    if baseline_failures:
        pd.DataFrame(baseline_failures).to_csv(REPORT_DIR / "baseline_failures.csv", index=False)
    display(pd.DataFrame(baseline_runs))
    if baseline_failures:
        display(pd.DataFrame(baseline_failures))
else:
    print("Baseline inference disabled.")

## 12. Load metric evidence across Kaggle sessions

The evaluator prefers files produced in the current `/kaggle/working` session and otherwise discovers prior metric CSVs attached under `/kaggle/input`.

In [ ]:
def collect_metric_files():
    current_files = sorted(METRIC_DIR.glob("segment_metrics_*.csv"))
    attached_files = sorted(Path("/kaggle/input").rglob("segment_metrics_*.csv"))
    selected = {}
    for path in attached_files:
        selected.setdefault(path.name, path)
    for path in current_files:
        selected[path.name] = path
    return list(selected.values())


metric_files = collect_metric_files()
print("Metric evidence files:", len(metric_files))
for path in metric_files:
    print(path)

## 13. Common evaluation, Wilcoxon tests, tables, and charts

`PRD_paper_pct` is the scientifically preferred formula. `PRD_repo_pct` reproduces the current repository implementation. Every model is recalculated by the same function, so the chosen column is consistent across methods.

Segment-level Wilcoxon is included for comparison with the repository style. Subject-level Wilcoxon is preferred for the main scientific interpretation because overlapping windows are correlated.

In [ ]:
if RUN_COMMON_EVALUATION:
    if not metric_files:
        raise FileNotFoundError("No segment_metrics_*.csv files found in working or attached inputs.")
    segment_metrics = pd.concat([pd.read_csv(path) for path in metric_files], ignore_index=True)
    identity_columns = ["model", "noise_version", "segment_index"]
    duplicate_count = int(segment_metrics.duplicated(identity_columns).sum())
    if duplicate_count:
        raise RuntimeError(f"Duplicate model/split/segment rows: {duplicate_count}")

    expected_models = ["MyDDPM"] + BASELINE_MODELS
    expected_counts = {
        row.noise_version: (
            int(row.test_segments)
            if MAX_TEST_SEGMENTS is None
            else min(int(row.test_segments), int(MAX_TEST_SEGMENTS))
        )
        for row in dataset_audit.itertuples(index=False)
    }
    observed_counts = segment_metrics.groupby(["model", "noise_version"]).size().to_dict()
    missing_or_wrong = []
    for model_name in expected_models:
        for noise_version, expected_count in expected_counts.items():
            observed_count = int(observed_counts.get((model_name, noise_version), 0))
            if observed_count != expected_count:
                missing_or_wrong.append({
                    "model": model_name,
                    "noise_version": noise_version,
                    "expected_segments": expected_count,
                    "observed_segments": observed_count,
                })
    unexpected_runs = sorted(
        set(observed_counts) -
        {(model_name, noise_version) for model_name in expected_models for noise_version in expected_counts}
    )
    if missing_or_wrong or unexpected_runs:
        raise RuntimeError(
            "Metric evidence is incomplete or inconsistent.\n"
            f"Missing/wrong runs: {missing_or_wrong}\nUnexpected runs: {unexpected_runs}"
        )

    prd_column = "PRD_paper_pct" if PRIMARY_PRD == "paper" else "PRD_repo_pct"
    display_metrics = ["SSD", "MAD_mV", prd_column, "CosSim", "RMSE_mV", "MAE_mV", "SNR_dB"]
    directions = {
        "SSD": "min", "MAD_mV": "min", prd_column: "min",
        "CosSim": "max", "RMSE_mV": "min", "MAE_mV": "min", "SNR_dB": "max",
    }

    summary_mean = segment_metrics.groupby("model")[display_metrics].mean()
    summary_std = segment_metrics.groupby("model")[display_metrics].std()
    model_order = summary_mean[prd_column].sort_values().index.tolist()
    summary = pd.concat({"mean": summary_mean, "std": summary_std}, axis=1).loc[model_order]
    display(summary.round(4))

    summary_flat = summary.copy()
    summary_flat.columns = [f"{stat}_{metric}" for stat, metric in summary_flat.columns]
    summary_flat.reset_index().to_csv(REPORT_DIR / "joint_metrics_summary.csv", index=False)
    segment_metrics.to_csv(REPORT_DIR / "joint_metrics_per_segment.csv", index=False)

    artifact_order = ["Clean", "BW", "MA", "BW+MA", "EM", "BW+EM", "MA+EM", "BW+MA+EM"]
    artifact_summary = (
        segment_metrics.groupby(["model", "artifact"])
        .agg(
            segments=("SNR_dB", "size"),
            SNR_mean_dB=("SNR_dB", "mean"),
            PRD_mean_pct=(prd_column, "mean"),
            MAD_mean_mV=("MAD_mV", "mean"),
        )
        .reset_index()
    )
    artifact_summary.to_csv(REPORT_DIR / "joint_metrics_by_artifact.csv", index=False)

    def paired_wilcoxon(level):
        rows = []
        if "MyDDPM" not in set(segment_metrics["model"]):
            return pd.DataFrame()
        if level == "subject":
            source = segment_metrics.groupby(["model", "noise_version", "subject"])[display_metrics].mean().reset_index()
            pair_keys = ["noise_version", "subject"]
        else:
            source = segment_metrics
            pair_keys = ["noise_version", "segment_index"]
        reference = source[source["model"] == "MyDDPM"]
        for other_name in sorted(set(source["model"]) - {"MyDDPM"}):
            other = source[source["model"] == other_name]
            paired = reference.merge(other, on=pair_keys, suffixes=("_ref", "_other"))
            for metric in display_metrics:
                ref_values = paired[f"{metric}_ref"].to_numpy()
                other_values = paired[f"{metric}_other"].to_numpy()
                difference = ref_values - other_values
                statistic, p_value = (0.0, 1.0) if np.allclose(difference, 0) else stats.wilcoxon(ref_values, other_values)
                rows.append({
                    "level": level,
                    "reference": "MyDDPM",
                    "comparison": other_name,
                    "metric": metric,
                    "pairs": len(paired),
                    "statistic": statistic,
                    "p_value": p_value,
                })
        return pd.DataFrame(rows)

    wilcoxon_results = pd.concat(
        [paired_wilcoxon("segment"), paired_wilcoxon("subject")],
        ignore_index=True,
    )
    wilcoxon_results.to_csv(REPORT_DIR / "joint_wilcoxon.csv", index=False)
    display(wilcoxon_results[wilcoxon_results["level"] == "subject"].head(30))

    sns.set_theme(style="whitegrid", context="notebook")
    chart_specs = [
        ("SSD", "SSD — lower is better"),
        ("MAD_mV", "MAD (mV) — lower is better"),
        (prd_column, f"PRD (%) — {PRIMARY_PRD}, lower is better"),
        ("CosSim", "Cosine similarity — higher is better"),
        ("RMSE_mV", "RMSE (mV) — lower is better"),
        ("MAE_mV", "MAE (mV) — lower is better"),
        ("SNR_dB", "SNR (dB) — higher is better"),
    ]
    fig, axes = plt.subplots(4, 2, figsize=(15, 19))
    axes = axes.ravel()
    for axis, (metric, title) in zip(axes, chart_specs):
        means = summary_mean.loc[model_order, metric]
        deviations = summary_std.loc[model_order, metric]
        best_model = means.idxmin() if directions[metric] == "min" else means.idxmax()
        colors = ["#D6A72D" if model == best_model else ("#C95B35" if model == "MyDDPM" else "#356AA0") for model in model_order]
        positions = np.arange(len(model_order))
        axis.errorbar(means, positions, xerr=deviations, fmt="none", ecolor="#9AA4AE", alpha=0.65, capsize=2)
        axis.scatter(means, positions, c=colors, s=58, edgecolor="#263238", linewidth=0.5, zorder=3)
        axis.set_yticks(positions, model_order)
        axis.set_title(title)
        axis.set_xlabel("Mean ± 1 SD across ECG segments")
        axis.grid(axis="x", color="#E3E7EB")
        axis.grid(axis="y", visible=False)
        axis.invert_yaxis()
    axes[-1].axis("off")
    fig.suptitle("Research DDPM vs published denoisers — shared RMN test set (nv1 + nv2)", fontsize=15, y=1.01)
    fig.tight_layout()
    fig.savefig(REPORT_DIR / "joint_metric_comparison.png", dpi=180, bbox_inches="tight")
    plt.show()

    snr_matrix = artifact_summary.pivot(index="model", columns="artifact", values="SNR_mean_dB")
    snr_matrix = snr_matrix.reindex(index=model_order, columns=artifact_order)
    fig, axis = plt.subplots(figsize=(13, max(5, 0.55 * len(model_order))))
    sns.heatmap(
        snr_matrix, annot=True, fmt=".2f", cmap="YlGnBu", linewidths=0.5,
        cbar_kws={"label": "Mean SNR (dB)"}, ax=axis,
    )
    axis.set_title("Mean SNR by model and artifact composition — nv1 + nv2")
    axis.set_xlabel("Artifact composition")
    axis.set_ylabel("Model")
    fig.tight_layout()
    fig.savefig(REPORT_DIR / "joint_snr_by_artifact.png", dpi=180, bbox_inches="tight")
    plt.show()

    full_test_expected = MAX_TEST_SEGMENTS is None
    validation_report = {
        "rows": len(segment_metrics),
        "models": sorted(segment_metrics["model"].unique().tolist()),
        "noise_versions": sorted(segment_metrics["noise_version"].unique().tolist()),
        "duplicate_identity_rows": duplicate_count,
        "all_metrics_finite": bool(np.isfinite(segment_metrics[display_metrics].to_numpy()).all()),
        "primary_prd": PRIMARY_PRD,
        "official_full_test_requested": full_test_expected,
    }
    (REPORT_DIR / "validation_report.json").write_text(json.dumps(validation_report, indent=2), encoding="utf-8")
    print(json.dumps(validation_report, indent=2))
else:
    print("Common evaluation disabled.")

## 14. Package outputs and next run instructions

During a train-only run, Kaggle persists only the four `.pth` files under `/kaggle/working/checkpoints`; the cloned repository, RMN datasets, audit CSV, and training log stay under `/kaggle/temp` and disappear when the session ends. Always save a Kaggle notebook version after training. To continue, attach that version's output under **Add Input** and run this notebook again. The resume logic searches for the exact `*_last.pth` names and rejects incompatible configuration metadata.

In [ ]:
manifest = {
    "created_at_utc": pd.Timestamp.utcnow().isoformat(),
    "config": CONFIG,
    "phase_switches": {
        "run_data_prep": RUN_DATA_PREP,
        "run_train": RUN_TRAIN,
        "train_splits": TRAIN_SPLITS,
        "run_research_inference": RUN_RESEARCH_INFERENCE,
        "run_baseline_inference": RUN_BASELINE_INFERENCE,
        "run_common_evaluation": RUN_COMMON_EVALUATION,
    },
    "max_test_segments": MAX_TEST_SEGMENTS,
    "official_result": MAX_TEST_SEGMENTS is None,
    "primary_prd": PRIMARY_PRD,
}
(RESULT_ROOT / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
if TRAIN_ONLY:
    print("Train-only storage policy: only /kaggle/working/checkpoints/*.pth is persisted.")
else:
    print("Evaluation artifacts:", RESULT_ROOT)

print("\nCheckpoints:")
for path in sorted(CHECKPOINT_DIR.glob("*.pth")):
    print(path.name, f"{path.stat().st_size / 1024**2:.1f} MB")
print("\nReports:")
for path in sorted(REPORT_DIR.glob("*")):
    print(path.name)

## Checks and interpretation rules

- A valid official run must use `MAX_TEST_SEGMENTS=None` for every model.
- The two research checkpoints must be trained independently on their matching `Dataset_1/2.pkl` files.
- Do not compare a BW-only legacy research checkpoint with the RMN baseline table.
- Report both denoising quality and inference cost. The research U-Net has approximately 11.5M parameters and is evaluated repeatedly across DDIM steps; FGDAE is substantially smaller.
- Mean ± SD is variation across ECG segments, not a confidence interval.
- Prefer subject-level paired Wilcoxon for the main scientific claim because overlapping segments are correlated; retain segment-level results for repository-style reproduction.
- Keep `PRD_paper_pct` and `PRD_repo_pct` explicitly separated. Never mix formulas across models.
- Results from a finite `MAX_TEST_SEGMENTS` value are diagnostics only.